In [1]:
using Random, Distributions, Statistics, Printf, DelimitedFiles, Dates
using LinearAlgebra
using StatsBase
using QuantileRegressions
using Plots
const bb = 120 
const aa = 40
const N  = 2_701_767
const I0 = 3
const S0 = 2_701_767 - 3 
const n_iter = 1_000_000
include("functions.jl")
Random.seed!(2025)


Istar_obs = [
2, 6, 11, 14, 17, 23, 31, 38, 43, 46, 74, 91, 119, 138, 193, 255, 257,
324, 372, 412, 422, 407, 411, 450, 408, 394, 371, 416, 425, 388, 387,
369, 386, 365, 328, 314, 335, 298, 323, 300, 280, 285, 273, 254, 253,
211, 209, 232, 203, 217, 199, 206, 217, 182, 173, 176, 154, 166, 157
]
tau = length(Istar_obs)

model_tag_sym = :exponential

KMAX_UPPER = 30  


# Fixed output dir
out_dir = "output"
isdir(out_dir) || mkpath(out_dir)

header_cont = []
if model_tag_sym === :memoryless
    global header_cont = ["beta", "alpha", "gamma"]
elseif model_tag_sym === :powerlaw
    global header_cont = ["beta", "alpha", "gamma", "lambda_P"]
elseif model_tag_sym === :exponential
    global header_cont = ["beta", "alpha", "gamma", "lambda_E"]
elseif model_tag_sym === :reciprocal
    global header_cont = ["beta", "alpha", "gamma", "lambda_R"] 
elseif model_tag_sym === :sliding
    global header_cont = ["beta", "alpha", "gamma"]            
else
    error("Unknown model tag: $(model_tag_sym)")
end

c = 3

@info "[$(String(model_tag_sym))_model] Fitting chain $(c) (tau=$tau)"

Random.seed!(2025 + c)
initθ_chain = initθ_for_chain(model_tag_sym) 
t0 = Dates.now()

try
    samples, loglik_aug_vecs =
        mcmc_one_chain_with_Rstar!(Istar_obs, N,S0, I0;
            fit_mech=model_tag_sym,
            n_iter=n_iter,
            initθ=initθ_chain,
            KMAX_UPPER=KMAX_UPPER)

    if size(samples, 1) != n_iter
        error("Chain $c did not complete all iterations.")
    end

    # Save samples
    samples_filename = "samples_chain_$(c).csv"
    write_csv(joinpath(out_dir, samples_filename), header_cont, samples)

    # Save per-time log-likelihoods (thinned & post-burnin inside mcmc)
    loglik_filename = "loglik_chain_$(c).csv"
    write_csv(joinpath(out_dir, loglik_filename), ["loglik"], hcat(loglik_aug_vecs))
    el = Dates.value(Dates.now() - t0) / 1000
catch err
    el = Dates.value(Dates.now() - t0) / 1000
end

@info "Chain completed -> output dir: $out_dir"


[ Info: [exponential_model] Fitting chain 3 (tau=59)
[ Info: [exponential] iter 1000/1000000 elapsed=5.6s, rate=0.184, mean=[0.649, 0.00344, 0.841, 0.804], std=[0.0401, 0.000261, 0.1349, 0.0149] [ADAPT]
[ Info: [exponential] iter 2000/1000000 elapsed=10.3s, rate=0.168, mean=[0.686, 0.00242, 0.871, 0.752], std=[0.0455, 0.000960, 0.1005, 0.0523] [ADAPT]
[ Info: [exponential] iter 3000/1000000 elapsed=14.2s, rate=0.153, mean=[0.707, 0.00200, 0.882, 0.730], std=[0.0459, 0.000954, 0.0836, 0.0529] [ADAPT]
[ Info: [exponential] iter 4000/1000000 elapsed=18.2s, rate=0.144, mean=[0.728, 0.00176, 0.898, 0.704], std=[0.0555, 0.000906, 0.0779, 0.0639] [ADAPT]
[ Info: [exponential] iter 5000/1000000 elapsed=22.1s, rate=0.138, mean=[0.750, 0.00165, 0.907, 0.650], std=[0.0633, 0.000848, 0.0718, 0.1261] [ADAPT]
[ Info: [exponential] iter 6000/1000000 elapsed=26.0s, rate=0.133, mean=[0.764, 0.00186, 0.899, 0.558], std=[0.0644, 0.000888, 0.0676, 0.2162] [ADAPT]
[ Info: [exponential] iter 7000/1000000 el